# Batch Expansion & Azure Classification

This notebook reads all `classified_index` + `stored_results` data from the source DB,
expands every matched snippet to its full surrounding clause **and** classifies it in a
single LLM call (the LLM decides clause boundaries instead of regex), then writes
everything into a new database named `icat-v2-beta`.

Supports checkpoint/resume so the notebook can be stopped and restarted without re-processing.

In [1]:
# Cell 1: Imports & Configuration
import sys
import os
import json
import time
from datetime import datetime

import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv()

import insert_data
import pipelineoperation

CHECKPOINT_FILE = "checkpoint_v2_beta.json"
BATCH_SIZE = 50
TARGET_DB_NAME = "icat-v2-beta"
AZURE_INFERENCE_MODEL = os.environ.get("AZURE_INFERENCE_MODEL")

print("Configuration loaded.")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  TARGET_DB_NAME = {TARGET_DB_NAME}")
print(f"  CHECKPOINT_FILE = {CHECKPOINT_FILE}")
print(f"  MODEL_NAME = {AZURE_INFERENCE_MODEL}")

Configuration loaded.
  BATCH_SIZE = 50
  TARGET_DB_NAME = icat-v2-beta
  CHECKPOINT_FILE = checkpoint_v2_beta.json
  MODEL_NAME = Llama-3.3-70B-Instruct


In [2]:
# Cell 2: Database Connections

# Source connection (uses env vars as-is)
source_conn = insert_data.create_connection()
assert source_conn is not None, "Failed to connect to source database"
print(f"Connected to source DB: {os.environ.get('DB_NAME')}")

# Create target database if it doesn't exist
admin_uri = (
    f"host={insert_data.DB_HOST} dbname={insert_data.DB_NAME} "
    f"user={insert_data.DB_USER} password={insert_data.DB_PASS} "
    f"sslmode={insert_data.DB_SSLMODE}"
)
admin_conn = psycopg.connect(admin_uri, autocommit=True)
with admin_conn.cursor() as cur:
    try:
        cur.execute(f'CREATE DATABASE "{TARGET_DB_NAME}"')
        print(f"Created database '{TARGET_DB_NAME}'")
    except psycopg.errors.DuplicateDatabase:
        print(f"Database '{TARGET_DB_NAME}' already exists")
admin_conn.close()

# Connect to target database
target_uri = (
    f"host={insert_data.DB_HOST} dbname={TARGET_DB_NAME} "
    f"user={insert_data.DB_USER} password={insert_data.DB_PASS} "
    f"sslmode={insert_data.DB_SSLMODE}"
)
target_conn = psycopg.connect(target_uri)
print(f"Connected to target DB: {TARGET_DB_NAME}")

# Initialize schema in target
insert_data.initialize_db(target_conn)
print("Target DB schema initialized.")

Connected to source DB: icatv1
Database 'icat-v2-beta' already exists
Connected to target DB: icat-v2-beta
Database initialized
Target DB schema initialized.


In [ ]:
# Cell 3: Test Run — Randomised Sample Across Queries
# Combined pipeline: expand + classify + extract discussion, all via LLM.
# Includes confidence scores and sentiment confidence.
# Exports full (untruncated) results to a timestamped CSV.

import random
import importlib
importlib.reload(pipelineoperation)
from pipelineoperation import expand_and_classify_with_azure, extract_discussion_with_azure

# Pick 5 random queries that have non-empty matching data
with source_conn.cursor() as cur:
    cur.execute("""
        SELECT DISTINCT searchquery FROM classified_index
        WHERE LENGTH(matching_columns) > 2 OR LENGTH(matching_indents) > 2
    """)
    all_queries = [r[0] for r in cur.fetchall()]

test_queries = random.sample(all_queries, min(5, len(all_queries)))
print(f"Randomly selected test queries: {test_queries}")

# Fetch all candidate rows per query, then randomly sample up to 4
test_rows = []
with source_conn.cursor() as cur:
    for q in test_queries:
        cur.execute("""
            SELECT doc_id, title, searchquery, matching_indents, matching_columns
            FROM classified_index WHERE searchquery = %s
            AND (LENGTH(matching_columns) > 2 OR LENGTH(matching_indents) > 2)
        """, (q,))
        candidates = cur.fetchall()
        sampled = random.sample(candidates, min(4, len(candidates)))
        test_rows.extend(sampled)

print(f"Randomly sampled {len(test_rows)} test rows")

# Prefetch Doc_Text for these docs
test_doc_ids = list({r[0] for r in test_rows})
test_doc_texts = {}
with source_conn.cursor() as cur:
    for doc_id in test_doc_ids:
        cur.execute("SELECT Doc_Text FROM stored_results WHERE Doc_ID = %s", (doc_id,))
        row = cur.fetchone()
        if row and row[0]:
            test_doc_texts[doc_id] = row[0]

print(f"Prefetched doc text for {len(test_doc_texts)} docs")

# Process each row: expand + classify + extract discussion
results_full = []  # full details for CSV
results_display = []  # truncated for display

for row in test_rows:
    doc_id, title, query, mi_raw, mc_raw = row
    doc_text = test_doc_texts.get(doc_id, '')
    mc = insert_data._safe_eval(mc_raw)
    mi = insert_data._safe_eval(mi_raw)

    # Expand + classify each snippet (one LLM call per snippet)
    mc_results = [expand_and_classify_with_azure(doc_text, s) for s in mc]
    mi_results = [expand_and_classify_with_azure(doc_text, s) for s in mi]

    expanded_mc = [r['clause_text'] for r in mc_results]
    expanded_mi = [r['clause_text'] for r in mi_results]
    classified_mc = [r['clause_text'] for r in mc_results if r['is_contract_clause']]
    classified_mi = [r['clause_text'] for r in mi_results if r['is_contract_clause']]

    # Per-snippet confidence and reasoning
    all_results = mc_results + mi_results
    classified_results = [r for r in all_results if r['is_contract_clause']]
    all_confidences = [r['classification_confidence'] for r in all_results]
    all_reasonings = [r['classification_reasoning'] for r in all_results]
    if classified_results:
        avg_conf = sum(r['classification_confidence'] for r in classified_results) / len(classified_results)
    elif all_results:
        avg_conf = sum(r['classification_confidence'] for r in all_results) / len(all_results)
    else:
        avg_conf = 0.0

    # Extract court discussion if any clauses classified as contract
    all_classified = classified_mc + classified_mi
    if all_classified and doc_text:
        disc_result = extract_discussion_with_azure(doc_text, all_classified[0])
        discussion = disc_result['discussion']
        sentiment = disc_result['sentiment']
        sent_conf = disc_result['sentiment_confidence']
    else:
        discussion = ''
        sentiment = ''
        sent_conf = 0.0

    # Full record (nothing truncated) for CSV
    results_full.append({
        'Query': query,
        'DocID': doc_id,
        'Title': title or '',
        'Doc_Text_Length': len(doc_text),
        'Snippets_MC': str(mc),
        'Snippets_MI': str(mi),
        'Expanded_MC': str(expanded_mc),
        'Expanded_MI': str(expanded_mi),
        'Classified_MC': str(classified_mc),
        'Classified_MI': str(classified_mi),
        'Num_Snippets_MC': len(mc),
        'Num_Snippets_MI': len(mi),
        'Num_Classified_MC': len(classified_mc),
        'Num_Classified_MI': len(classified_mi),
        'Avg_Classification_Confidence': round(avg_conf, 4),
        'All_Confidences': str(all_confidences),
        'All_Reasonings': str(all_reasonings),
        'Sentiment': sentiment,
        'Sentiment_Confidence': round(sent_conf, 4),
        'Discussion': discussion,
        'Is_Contract': 'Yes' if (classified_mc or classified_mi) else 'No',
        'IndianKanoon_URL': f'https://indiankanoon.org/doc/{doc_id}/',
    })

    # Truncated record for notebook display
    sample = (classified_mc[0][:200] if classified_mc else
              classified_mi[0][:200] if classified_mi else
              expanded_mc[0][:200] if expanded_mc else
              expanded_mi[0][:200] if expanded_mi else '\u2014')
    results_display.append({
        'Query': query,
        'DocID': doc_id,
        'Title': (title or '')[:50],
        'MC': len(mc),
        'MI': len(mi),
        'Cls MC': len(classified_mc),
        'Cls MI': len(classified_mi),
        'Cls Conf': round(avg_conf, 2),
        'Sentiment': sentiment,
        'Sent Conf': round(sent_conf, 2),
        'Discussion': (discussion or '')[:200],
        'Sample Clause': sample,
        'Contract?': 'Yes' if (classified_mc or classified_mi) else 'No',
    })

# Display summary table
df_display = pd.DataFrame(results_display)
display(df_display)

# Export full details to timestamped CSV
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = f"test_run_{timestamp}.csv"
df_full = pd.DataFrame(results_full)
df_full.to_csv(csv_path, index=False)
print(f"\nFull details exported to: {csv_path}")
print(f"  {len(results_full)} rows, {len(df_full.columns)} columns")
print(f"  Columns: {list(df_full.columns)}")
print("\n** Review the table above. Confidence scores should now appear. **")
print("** Do not proceed to full processing until satisfied. **")

Randomly selected test queries: ['arbitration', 'payment terms', 'governing law', 'non compete', 'Arbitration ']
Randomly sampled 20 test rows
Prefetched doc text for 20 docs


In [ ]:
# Cell 4: Copy Base Tables to Target

# Clear any aborted transaction state, reload module, ensure tables exist
target_conn.rollback()
import importlib
importlib.reload(insert_data)
insert_data.initialize_db(target_conn)

# Copy stored_results
with source_conn.cursor() as src_cur:
    src_cur.execute("SELECT Doc_ID, Title, Doc_Text, Doc_Blockquotes, Doc_Size FROM stored_results")
    stored_rows = src_cur.fetchall()

inserted_sr = 0
with target_conn.cursor() as tgt_cur:
    for row in stored_rows:
        tgt_cur.execute("""
            INSERT INTO stored_results (Doc_ID, Title, Doc_Text, Doc_Blockquotes, Doc_Size)
            VALUES (%s, %s, %s, %s, %s)
            ON CONFLICT (Doc_ID) DO NOTHING
        """, row)
        inserted_sr += tgt_cur.rowcount
    target_conn.commit()

print(f"stored_results: {len(stored_rows)} source rows, {inserted_sr} newly inserted into target")

# Copy search_queries
with source_conn.cursor() as src_cur:
    src_cur.execute("SELECT searchquery, dateandtime FROM search_queries")
    sq_rows = src_cur.fetchall()

with target_conn.cursor() as tgt_cur:
    for row in sq_rows:
        tgt_cur.execute("""
            INSERT INTO search_queries (searchquery, dateandtime)
            VALUES (%s, %s)
        """, row)
    target_conn.commit()

print(f"search_queries: {len(sq_rows)} rows copied to target")

In [ ]:
# Cell 4.5: Extract Court Metadata for stored_results
# Populates court_name, judgment_date, case_citation using Indian Kanoon API + LLM fallback.
# Has its own checkpoint file so it can be re-run independently.

import importlib
importlib.reload(pipelineoperation)
importlib.reload(insert_data)

import sys
sys.path.insert(0, '.')
from app import fetch_docmeta
from pipelineoperation import extract_metadata_with_azure

METADATA_CHECKPOINT = "checkpoint_metadata.json"
api_key = os.environ.get("API_KEY")
meta_headers = {'authorization': f"Token {api_key}"}

# Load metadata checkpoint
if os.path.exists(METADATA_CHECKPOINT):
    with open(METADATA_CHECKPOINT, 'r') as f:
        meta_ckpt = json.load(f)
    meta_processed = set(meta_ckpt.get('processed_doc_ids', []))
    print(f"Metadata checkpoint loaded: {len(meta_processed)} docs already processed")
else:
    meta_processed = set()
    print("No metadata checkpoint found. Starting fresh.")

# Find docs needing metadata
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT Doc_ID, Doc_Text FROM stored_results
        WHERE court_name IS NULL OR court_name = ''
    """)
    docs_needing_meta = cur.fetchall()

# Filter out already-checkpointed docs
docs_to_process = [(d, t) for d, t in docs_needing_meta if d not in meta_processed]
print(f"{len(docs_to_process)} docs need metadata ({len(docs_needing_meta)} total without, {len(meta_processed)} already done)")

meta_stats = {'api_success': 0, 'llm_fallback': 0, 'failed': 0}

for i, (doc_id, doc_text) in enumerate(docs_to_process):
    try:
        # Try API first
        meta = fetch_docmeta(doc_id, meta_headers)

        # If court_name still empty, use LLM fallback
        if not meta.get('court_name') and doc_text:
            meta = extract_metadata_with_azure(doc_text[:2000])
            meta_stats['llm_fallback'] += 1
        else:
            meta_stats['api_success'] += 1

        # Update stored_results
        with target_conn.cursor() as cur:
            cur.execute("""
                UPDATE stored_results
                SET court_name = %s, judgment_date = %s, case_citation = %s
                WHERE Doc_ID = %s
            """, (meta.get('court_name', ''), meta.get('judgment_date', ''),
                  meta.get('case_citation', ''), doc_id))

        meta_processed.add(doc_id)

        # Checkpoint every BATCH_SIZE docs
        if (i + 1) % BATCH_SIZE == 0:
            target_conn.commit()
            with open(METADATA_CHECKPOINT, 'w') as f:
                json.dump({'processed_doc_ids': list(meta_processed),
                           'last_updated': datetime.now().isoformat(),
                           'stats': meta_stats}, f, indent=2)
            print(f"  Metadata: {i + 1}/{len(docs_to_process)} processed "
                  f"(API: {meta_stats['api_success']}, LLM: {meta_stats['llm_fallback']})")

        time.sleep(0.2)  # rate limit

    except Exception as e:
        print(f"  Metadata error for {doc_id}: {e}")
        meta_stats['failed'] += 1
        meta_processed.add(doc_id)

# Final commit and checkpoint
target_conn.commit()
with open(METADATA_CHECKPOINT, 'w') as f:
    json.dump({'processed_doc_ids': list(meta_processed),
               'last_updated': datetime.now().isoformat(),
               'stats': meta_stats}, f, indent=2)

print(f"\nMetadata extraction complete:")
print(f"  API success: {meta_stats['api_success']}")
print(f"  LLM fallback: {meta_stats['llm_fallback']}")
print(f"  Failed: {meta_stats['failed']}")

In [ ]:
# Cell 5: Load Checkpoint

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        checkpoint = json.load(f)
    processed_ids = set(checkpoint.get('processed_ids', []))
    stats = checkpoint.get('stats', {
        'total_processed': 0,
        'total_expanded': 0,
        'total_classified_as_clause': 0,
        'total_discussions_extracted': 0,
        'total_errors': 0,
    })
    stats.setdefault('total_discussions_extracted', 0)
    print(f"Checkpoint loaded: {len(processed_ids)} rows already processed")
else:
    processed_ids = set()
    stats = {
        'total_processed': 0,
        'total_expanded': 0,
        'total_classified_as_clause': 0,
        'total_discussions_extracted': 0,
        'total_errors': 0,
    }
    print("No checkpoint found. Starting fresh.")

In [ ]:
# Cell 6: Fetch & Prepare Work Queue

# Fetch all classified_index rows from source
with source_conn.cursor() as cur:
    cur.execute("""
        SELECT doc_id, title, searchquery, matching_indents, matching_columns,
               matching_columns_after_classification, matching_indents_after_classification
        FROM classified_index
    """)
    all_rows = cur.fetchall()

print(f"Total classified_index rows in source: {len(all_rows)}")

# Prefetch all Doc_Text from stored_results
doc_texts = {}
with source_conn.cursor() as cur:
    cur.execute("SELECT Doc_ID, Doc_Text FROM stored_results")
    for row in cur.fetchall():
        if row[1]:
            doc_texts[row[0]] = row[1]

print(f"Prefetched Doc_Text for {len(doc_texts)} documents")

# Build work queue, filtering out already-processed rows
work_queue = []
for row in all_rows:
    doc_id, title, searchquery, mi_raw, mc_raw, mc_after_raw, mi_after_raw = row
    composite_key = f"{doc_id}__{searchquery}"
    if composite_key in processed_ids:
        continue
    work_queue.append({
        'doc_id': doc_id,
        'title': title,
        'searchquery': searchquery,
        'matching_indents': insert_data._safe_eval(mi_raw),
        'matching_columns': insert_data._safe_eval(mc_raw),
        'matching_columns_after_classification': insert_data._safe_eval(mc_after_raw),
        'matching_indents_after_classification': insert_data._safe_eval(mi_after_raw),
        'composite_key': composite_key,
    })

print(f"{len(work_queue)} rows to process ({len(processed_ids)} already checkpointed)")

In [ ]:
# Cell 7: Process — Expand, Classify & Extract Discussion (main loop)
# Per snippet: one LLM call for expand + classify (now with confidence scores).
# Per row (if contract clauses found): one LLM call for discussion extraction (now with sentiment confidence).
# Run Cell 5 (checkpoint) and Cell 6 (work queue) before this cell.

import importlib
importlib.reload(pipelineoperation)
from pipelineoperation import expand_and_classify_with_azure, extract_discussion_with_azure

def save_checkpoint():
    """Save current progress to checkpoint file."""
    checkpoint_data = {
        'processed_ids': list(processed_ids),
        'last_updated': datetime.now().isoformat(),
        'stats': stats,
    }
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint_data, f, indent=2)


def expand_classify_with_retry(doc_text, snippet, max_retries=3):
    """Combined expand+classify with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return expand_and_classify_with_azure(doc_text, snippet)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


def extract_discussion_with_retry(doc_text, clause, max_retries=3):
    """Extract discussion with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return extract_discussion_with_azure(doc_text, clause)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Discussion retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


insert_sql = """
    INSERT INTO classified_index(
        Doc_Id, Title, searchquery,
        matching_indents, matching_columns,
        matching_columns_after_classification,
        matching_indents_after_classification,
        expanded_columns, expanded_indents,
        expanded_columns_after_classification,
        expanded_indents_after_classification,
        extracted_discussion, sentiment,
        classification_confidence, classification_reasoning,
        sentiment_confidence
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (Doc_Id, searchquery) DO UPDATE SET
        Title = EXCLUDED.Title,
        matching_indents = EXCLUDED.matching_indents,
        matching_columns = EXCLUDED.matching_columns,
        matching_columns_after_classification = EXCLUDED.matching_columns_after_classification,
        matching_indents_after_classification = EXCLUDED.matching_indents_after_classification,
        expanded_columns = EXCLUDED.expanded_columns,
        expanded_indents = EXCLUDED.expanded_indents,
        expanded_columns_after_classification = EXCLUDED.expanded_columns_after_classification,
        expanded_indents_after_classification = EXCLUDED.expanded_indents_after_classification,
        extracted_discussion = EXCLUDED.extracted_discussion,
        sentiment = EXCLUDED.sentiment,
        classification_confidence = EXCLUDED.classification_confidence,
        classification_reasoning = EXCLUDED.classification_reasoning,
        sentiment_confidence = EXCLUDED.sentiment_confidence
"""

total_batches = (len(work_queue) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Starting processing: {len(work_queue)} rows in {total_batches} batches")

for batch_idx in range(0, len(work_queue), BATCH_SIZE):
    batch = work_queue[batch_idx:batch_idx + BATCH_SIZE]
    batch_num = batch_idx // BATCH_SIZE + 1

    for row in batch:
        try:
            doc_text = doc_texts.get(row['doc_id'], '')

            # Combined expand + classify for matching_columns snippets
            mc_results = []
            for s in row['matching_columns']:
                mc_results.append(expand_classify_with_retry(doc_text, s))
                time.sleep(0.1)

            # Combined expand + classify for matching_indents snippets
            mi_results = []
            for s in row['matching_indents']:
                mi_results.append(expand_classify_with_retry(doc_text, s))
                time.sleep(0.1)

            # Derive all column values from the combined results
            expanded_columns = [r['clause_text'] for r in mc_results]
            expanded_indents = [r['clause_text'] for r in mi_results]
            expanded_columns_after = [r['clause_text'] for r in mc_results if r['is_contract_clause']]
            expanded_indents_after = [r['clause_text'] for r in mi_results if r['is_contract_clause']]

            # Original snippet classification derived from the same LLM judgment
            mc_after = [s for s, r in zip(row['matching_columns'], mc_results) if r['is_contract_clause']]
            mi_after = [s for s, r in zip(row['matching_indents'], mi_results) if r['is_contract_clause']]

            # Compute average classification confidence across all classified results
            all_results = mc_results + mi_results
            classified_results = [r for r in all_results if r['is_contract_clause']]
            if classified_results:
                avg_confidence = sum(r['classification_confidence'] for r in classified_results) / len(classified_results)
            else:
                avg_confidence = sum(r['classification_confidence'] for r in all_results) / len(all_results) if all_results else 0.0
            first_reasoning = classified_results[0]['classification_reasoning'] if classified_results else (
                all_results[0]['classification_reasoning'] if all_results else ''
            )

            # Extract court discussion if any clauses classified as contract
            all_classified = expanded_columns_after + expanded_indents_after
            if all_classified and doc_text:
                disc_result = extract_discussion_with_retry(doc_text, all_classified[0])
                extracted_discussion = disc_result['discussion']
                sentiment = disc_result['sentiment']
                sentiment_conf = disc_result['sentiment_confidence']
                stats['total_discussions_extracted'] += 1
                time.sleep(0.1)
            else:
                extracted_discussion = ''
                sentiment = ''
                sentiment_conf = 0.0

            # Insert into target DB (16 columns)
            with target_conn.cursor() as tgt_cur:
                tgt_cur.execute(insert_sql, (
                    row['doc_id'],
                    row['title'],
                    row['searchquery'],
                    str(row['matching_indents']),
                    str(row['matching_columns']),
                    str(mc_after),
                    str(mi_after),
                    str(expanded_columns),
                    str(expanded_indents),
                    str(expanded_columns_after),
                    str(expanded_indents_after),
                    extracted_discussion,
                    sentiment,
                    str(round(avg_confidence, 3)),
                    first_reasoning,
                    str(round(sentiment_conf, 3)),
                ))

            # Track stats
            stats['total_expanded'] += len(expanded_columns) + len(expanded_indents)
            stats['total_classified_as_clause'] += (
                len(expanded_columns_after) + len(expanded_indents_after)
            )
            processed_ids.add(row['composite_key'])
            stats['total_processed'] += 1

        except Exception as e:
            print(f"  ERROR processing {row['composite_key']}: {e}")
            stats['total_errors'] += 1
            processed_ids.add(row['composite_key'])  # skip on future runs

    # Commit and checkpoint after each batch
    target_conn.commit()
    save_checkpoint()
    print(f"Batch {batch_num}/{total_batches} complete "
          f"({stats['total_processed']} total processed, "
          f"{stats['total_discussions_extracted']} discussions)")

In [ ]:
# Cell 10: Verification

# Row counts in target
with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    target_ci_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results")
    target_sr_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM search_queries")
    target_sq_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM classified_index WHERE LENGTH(extracted_discussion) > 0")
    target_disc_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM classified_index WHERE classification_confidence IS NOT NULL AND classification_confidence != ''")
    target_conf_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results WHERE court_name IS NOT NULL AND court_name != ''")
    target_meta_count = cur.fetchone()[0]

# Row counts in source
with source_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    source_ci_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results")
    source_sr_count = cur.fetchone()[0]

print("=== Row Count Comparison ===")
print(f"  classified_index:  source={source_ci_count}  target={target_ci_count}")
print(f"  stored_results:    source={source_sr_count}  target={target_sr_count}")
print(f"  search_queries:    target={target_sq_count}")
print(f"  with discussion:   target={target_disc_count}")
print(f"  with confidence:   target={target_conf_count}")
print(f"  with court meta:   target={target_meta_count}")

# Sentiment distribution with confidence
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT sentiment, COUNT(*),
               AVG(CASE WHEN sentiment_confidence != '' THEN CAST(sentiment_confidence AS FLOAT) ELSE NULL END)
        FROM classified_index
        WHERE LENGTH(extracted_discussion) > 0
        GROUP BY sentiment
    """)
    sentiment_dist = cur.fetchall()

if sentiment_dist:
    print("\n=== Sentiment Distribution (with avg confidence) ===")
    for sentiment, count, avg_conf in sentiment_dist:
        avg_str = f"{avg_conf:.2f}" if avg_conf else "N/A"
        print(f"  {sentiment or '(empty)'}: {count} (avg confidence: {avg_str})")

# Classification confidence distribution
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT 
            CASE 
                WHEN CAST(classification_confidence AS FLOAT) >= 0.8 THEN 'high (>=0.8)'
                WHEN CAST(classification_confidence AS FLOAT) >= 0.5 THEN 'medium (0.5-0.8)'
                ELSE 'low (<0.5)'
            END as conf_bucket,
            COUNT(*)
        FROM classified_index
        WHERE classification_confidence IS NOT NULL AND classification_confidence != ''
        GROUP BY conf_bucket
    """)
    conf_dist = cur.fetchall()

if conf_dist:
    print("\n=== Classification Confidence Distribution ===")
    for bucket, count in conf_dist:
        print(f"  {bucket}: {count}")

# Show sample rows with all new fields
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT ci.doc_id, ci.searchquery, ci.sentiment,
               ci.classification_confidence, ci.classification_reasoning,
               ci.sentiment_confidence,
               sr.court_name, sr.judgment_date, sr.case_citation
        FROM classified_index ci
        LEFT JOIN stored_results sr ON ci.doc_id = sr.Doc_ID
        WHERE LENGTH(ci.extracted_discussion) > 0
        LIMIT 10
    """)
    verification_rows = cur.fetchall()

if verification_rows:
    print("\n=== Sample Rows with All New Fields ===")
    for r in verification_rows:
        print(f"  DocID: {r[0]}, Query: {r[1]}")
        print(f"    Sentiment: {r[2]}, Confidence: {r[5]}")
        print(f"    Classification Confidence: {r[3]}, Reasoning: {(r[4] or '')[:80]}")
        print(f"    Court: {r[6]}, Date: {r[7]}, Citation: {(r[8] or '')[:60]}")
        print()
else:
    print("\nNo rows with extracted discussions found.")

# Close connections
source_conn.close()
target_conn.close()
print("\nBoth database connections closed.")